# Video Performance Feature Engineering

Creates the video-level dataset used by the Video Performance dashboard. Duplicate exploratory cells were consolidated and the final bins match the Power BI visuals.

## 1. Load and validate the video-level dataset

Each row represents one unique video.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")
INPUT_FILE = DATA_DIR / "video_features.csv"
OUTPUT_FILE = DATA_DIR / "video_features_powerbi.csv"

item_df = pd.read_csv(INPUT_FILE)
print(f"Rows: {len(item_df):,}")
item_df.head()


Rows: 449,472


,item_id,item_city,music_id,publish_time,total_views,like_count
0,0,24.0,220.0,2019-10-27,24,0
1,1,63.0,574.0,2019-10-27,1309,5
2,3,7.0,26289.0,2019-10-27,2,0
3,4,146.0,162.0,2019-10-27,613,3
4,7,33.0,540.0,2019-10-09,2,0


In [2]:
required_columns = [
    "item_id", "item_city", "music_id", "publish_time",
    "total_views", "like_count"
]
missing_columns = [col for col in required_columns if col not in item_df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

if not item_df["item_id"].is_unique:
    raise ValueError("The input must contain one row per video.")


## 2. Prepare date and engagement features

In [3]:
item_df["publish_time"] = pd.to_datetime(item_df["publish_time"], errors="coerce")

item_df["publish_year"] = item_df["publish_time"].dt.year
item_df["publish_month"] = item_df["publish_time"].dt.month
item_df["publish_day"] = item_df["publish_time"].dt.day
item_df["publish_weekday"] = item_df["publish_time"].dt.day_name()
item_df["publish_year_month"] = item_df["publish_time"].dt.to_period("M").astype("string")

item_df["like_rate"] = np.where(
    item_df["total_views"] > 0,
    item_df["like_count"] / item_df["total_views"],
    0
).round(4)

item_df["has_like"] = (item_df["like_count"] > 0).astype("int8")


## 3. Create distribution bins

In [4]:
def add_ordered_bin(data, source, bins, labels, output):
    data[output] = pd.cut(
        data[source],
        bins=bins,
        labels=labels,
        right=True,
        include_lowest=True
    )
    order_map = {label: order for order, label in enumerate(labels, start=1)}
    data[f"{output}_order"] = (
        data[output].astype("string").map(order_map).astype("Int64")
    )

add_ordered_bin(
    item_df, "total_views",
    [0, 1, 2, 4, np.inf],
    ["0 view", "1–2 views", "3–4 views", "5+ views"],
    "view_bin"
)

add_ordered_bin(
    item_df, "like_count",
    [-1, 0, 1, np.inf],
    ["0 likes", "1 like", "2+ likes"],
    "like_bin"
)


## 4. Validate and export

In [5]:
bin_columns = ["view_bin", "like_bin"]
order_columns = ["view_bin_order", "like_bin_order"]

for column in bin_columns:
    item_df[column] = item_df[column].astype("string")

assert item_df["item_id"].is_unique
assert item_df[bin_columns].notna().all().all()
assert item_df[order_columns].notna().all().all()
assert item_df["like_rate"].between(0, 1).all()

output_columns = [
    "item_id", "item_city", "music_id", "publish_time",
    "publish_year", "publish_month", "publish_day",
    "publish_weekday", "publish_year_month",
    "total_views", "like_count", "like_rate", "has_like",
    "view_bin", "view_bin_order", "like_bin", "like_bin_order"
]

video_features_powerbi = item_df[output_columns].copy()
video_features_powerbi.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Exported {len(video_features_powerbi):,} videos to {OUTPUT_FILE}")
print(f"Total likes: {video_features_powerbi['like_count'].sum():,}")
video_features_powerbi.head()


Exported 449,472 videos to ../data/processed/video_features_powerbi.csv
Total likes: 16,773


,item_id,item_city,music_id,publish_time,publish_year,publish_month,publish_day,publish_weekday,publish_year_month,total_views,like_count,like_rate,has_like,view_bin,view_bin_order,like_bin,like_bin_order
0,0,24.0,220.0,2019-10-27,2019,10,27,Sunday,2019-10,24,0,0.0000,0,5+ views,4,0 likes,1
1,1,63.0,574.0,2019-10-27,2019,10,27,Sunday,2019-10,1309,5,0.0038,1,5+ views,4,2+ likes,3
2,3,7.0,26289.0,2019-10-27,2019,10,27,Sunday,2019-10,2,0,0.0000,0,1–2 views,2,0 likes,1
3,4,146.0,162.0,2019-10-27,2019,10,27,Sunday,2019-10,613,3,0.0049,1,5+ views,4,2+ likes,3
4,7,33.0,540.0,2019-10-09,2019,10,9,Wednesday,2019-10,2,0,0.0000,0,1–2 views,2,0 likes,1
